In [1]:
# %pip install matplotlib seaborn numpy


In [2]:
# EDA Notebook (Artifact-Compliant)
# Requirements:
# - Uses matplotlib (no seaborn)
# - Saves plots to ../plots/
# - Generates >= 3 plots
# - Reads input from ingestion output files

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
# Set Up Paths
# Create directory for plots if it doesn't exist
PLOTS_DIR = Path("../plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
# Define paths for train and test data
TRAIN_PATH = Path("../data/processed/cmi_sensor_data/train_raw.csv")
TEST_PATH = Path("../data/processed/cmi_sensor_data/test_raw.csv")

In [5]:
# Check if data files exist
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Train data not found at {TRAIN_PATH}. Run data ingestion first.")
if not TEST_PATH.exists():
    raise FileNotFoundError(f"Test data not found at {TEST_PATH}. Run data ingestion first.")

In [6]:
# Load Data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nFirst 5 rows of train data:")
# display(train_df.head())

Train shape: (574945, 341)
Test shape: (107, 336)

First 5 rows of train data:


In [7]:
# Basic Data Structure Checks
print("\nData types in train data:")
print(train_df.dtypes.value_counts())

print("\nTrain data info:")
# train_df.info()


Data types in train data:
float64    332
str          8
int64        1
Name: count, dtype: int64

Train data info:


In [8]:
# Check if the target column exists
TARGET_COL = "gesture"
if TARGET_COL not in train_df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found in train data.")

In [9]:
# Missing Values Analysis
missing_counts = train_df.isna().sum()
missing_nonzero = missing_counts[missing_counts > 0].sort_values(ascending=False)
print("\nNumber of columns with missing values:", len(missing_nonzero))
print("Top 30 columns with missing values:")
# display(missing_nonzero.head(30))


Number of columns with missing values: 329
Top 30 columns with missing values:


In [10]:
# Plot 1: Top 20 columns with missing values (if any)
if len(missing_nonzero) > 0:
    top_missing = missing_nonzero.head(20)
    plt.figure(figsize=(10, 6))
    plt.bar(top_missing.index.astype(str), top_missing.values)
    plt.title("Top 20 Columns by Missing Values (Train)")
    plt.xlabel("Column")
    plt.ylabel("Missing Count")
    plt.xticks(rotation=70, ha="right")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "eda_missing_values_top20.png", dpi=200)
    plt.close()

    # plt.show()
    print("Plot saved: Top 20 columns with missing values.")
else:
    print("No missing values found in train data.")

Plot saved: Top 20 columns with missing values.


In [11]:
# Target Distribution Analysis
gesture_counts = train_df[TARGET_COL].value_counts()
print("\nNumber of unique gestures:", train_df[TARGET_COL].nunique())
print("Top 30 gesture counts:")
# display(gesture_counts.head(30))

# Plot 2: Distribution of gesture classes (top 25)
top_k = 25 if len(gesture_counts) > 25 else len(gesture_counts)
top_gestures = gesture_counts.head(top_k)

plt.figure(figsize=(12, 6))
plt.bar(top_gestures.index.astype(str), top_gestures.values)
plt.title(f"Distribution of Gesture Classes (Top {top_k})")
plt.xlabel("Gesture")
plt.ylabel("Count")
plt.xticks(rotation=70, ha="right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eda_gesture_distribution_top.png", dpi=200)
plt.close()

# plt.show()
print("Plot saved: Gesture distribution.")


Number of unique gestures: 18
Top 30 gesture counts:
Plot saved: Gesture distribution.


In [12]:
# Calculate imbalance ratio
imbalance_ratio = gesture_counts.max() / max(1, gesture_counts.min())
print(f"Imbalance ratio (max/min class count): {imbalance_ratio:.2f}")

Imbalance ratio (max/min class count): 5.94


In [13]:
# Numeric Sensor Data Summary
# Exclude target column and select only numeric columns
X = train_df.drop(columns=[TARGET_COL], errors="ignore")
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("\nNumber of numeric feature columns:", len(numeric_cols))
print("Sample numeric columns:", numeric_cols[:10])


Number of numeric feature columns: 333
Sample numeric columns: ['sequence_counter', 'acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2']


In [14]:
# Plot 3: Distributions of selected numeric sensor columns (first 5)
if len(numeric_cols) >= 5:
    for col in numeric_cols[:5]:
        data = train_df[col].dropna()
        # Sample data if too large
        if len(data) > 200_000:
            data = data.sample(200_000, random_state=42)

        plt.figure(figsize=(10, 6))
        plt.hist(data.values, bins=60)
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"eda_dist_{col}.png", dpi=200)
        plt.close()
        # plt.show()
        print(f"Plot saved: Distribution of {col}.")
else:
    print("No numeric columns found to plot.")

Plot saved: Distribution of sequence_counter.
Plot saved: Distribution of acc_x.
Plot saved: Distribution of acc_y.
Plot saved: Distribution of acc_z.
Plot saved: Distribution of rot_w.


In [15]:
# Correlation Heatmap (Subset)
# Use a small subset of numeric features for correlation analysis
corr_subset = numeric_cols[:20]

if len(corr_subset) >= 3:
    corr_data = train_df[corr_subset].dropna()
    if len(corr_data) > 50_000:
        corr_data = corr_data.sample(50_000, random_state=42)

    corr_matrix = corr_data.corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    plt.imshow(corr_matrix.values, aspect="auto", cmap="coolwarm")
    plt.title("Correlation Heatmap (First 20 Numeric Features)")
    plt.xticks(range(len(corr_subset)), corr_subset, rotation=90)
    plt.yticks(range(len(corr_subset)), corr_subset)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "eda_correlation_heatmap_first20.png", dpi=200)
    plt.close()
    print("Plot saved: Correlation heatmap.")
else:
    print("Not enough numeric columns for correlation heatmap.")

Plot saved: Correlation heatmap.


In [16]:
# Train vs. Test Structure Check
train_sensor_cols = [
    c for c in train_df.columns
    if c.startswith(("acc_", "rot_", "thm_", "tof_"))
]

test_sensor_cols = [
    c for c in test_df.columns
    if c.startswith(("acc_", "rot_", "thm_", "tof_"))
]

print("Sensor columns match:",
      set(train_sensor_cols) == set(test_sensor_cols))

print("\nEDA complete. All plots saved in:", PLOTS_DIR.resolve())

Sensor columns match: True

EDA complete. All plots saved in: /Users/ujjwaldahiya/Desktop/capstone-another-check/2026-winter-capstone-project-2026winter-capstone-group-5/plots


In [19]:
# Correlation Heatmap (Subset)
# Use a small subset of numeric features for correlation analysis
corr_subset = numeric_cols[:20]

if len(corr_subset) >= 3:
    corr_data = train_df[corr_subset].dropna()
    if len(corr_data) > 50_000:
        corr_data = corr_data.sample(50_000, random_state=42)

    corr_matrix = corr_data.corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    plt.imshow(corr_matrix.values, aspect="auto", cmap="coolwarm")
    plt.title("Correlation Heatmap (First 20 Numeric Features)")
    plt.xticks(range(len(corr_subset)), corr_subset, rotation=90)
    plt.yticks(range(len(corr_subset)), corr_subset)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "eda_correlation_heatmap_first20.png", dpi=200)
    plt.close()
    plt.show()
    print("Plot saved: Correlation heatmap.")
else:
    print("Not enough numeric columns for correlation heatmap.")

Plot saved: Correlation heatmap.


In [18]:
# Train vs. Test Structure Check
train_sensor_cols = [
    c for c in train_df.columns
    if c.startswith(("acc_", "rot_", "thm_", "tof_"))
]

test_sensor_cols = [
    c for c in test_df.columns
    if c.startswith(("acc_", "rot_", "thm_", "tof_"))
]

print("Sensor columns match:",
      set(train_sensor_cols) == set(test_sensor_cols))

# - The dataset consists of multivariate sensor readings collected over time.
# - The target variable `gesture` contains multiple distinct classes.
# - No major structural differences were observed between train and test datasets.
# - Sensor values vary significantly across features, indicating the need for scaling.
# - Some gestures appear more frequently than others, suggesting mild class imbalance.
# - No preprocessing or feature engineering has been applied at this stage.

Sensor columns match: True
